# NQ100 売買ルールバックテスト

Google Drive上のCSVを読み込み、東京時間の市場状態AIと売買シミュレーションを検証するためのColabノートブックです。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
import sys
from pathlib import Path

REPO_DIR = Path('/content/NQ100')
REPO_URL = 'https://github.com/hon-daisuki/NQ100.git'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'reset', '--hard', 'origin/main'], cwd=REPO_DIR, check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))


In [ ]:
import pandas as pd
import numpy as np

from src.labels import add_future_return_label, add_direction_label
from src.backtest import build_long_short_returns, equity_curve, summarize_returns, yearly_returns
from src.metrics import describe_time_range, missing_summary

## データ読み込み

In [ ]:
DATA_ROOT = Path('/content/drive')
CSV_CANDIDATES = [
    'USTEC_features_all.csv',
    'USTEC_1hour_features.csv',
]
PREFERRED_DIRS = [
    Path('/content/drive/MyDrive/CFD機械学習/backtest_ready'),
    Path('/content/drive/MyDrive/CFD機械学習'),
    Path('/content/drive/MyDrive'),
    DATA_ROOT,
]

def find_features_csv():
    checked = []
    for folder in PREFERRED_DIRS:
        for name in CSV_CANDIDATES:
            candidate = folder / name
            checked.append(str(candidate))
            if candidate.exists():
                return candidate

    matches = []
    for name in CSV_CANDIDATES:
        matches.extend(DATA_ROOT.rglob(name))
    if matches:
        return sorted(matches, key=lambda x: len(str(x)))[0]

    raise FileNotFoundError(
        '特徴量CSVが見つかりません。Driveに対象CSVがあるか、共有フォルダをMyDriveにショートカット追加してください。\n'
        + '探した候補:\n' + '\n'.join(checked)
    )

csv_path = find_features_csv()
print(f'Using CSV: {csv_path}')

df = pd.read_csv(csv_path)
df['日時'] = pd.to_datetime(df['日時'])
df = df.sort_values('日時').reset_index(drop=True)

describe_time_range(df)

In [ ]:
missing_summary(df)

## 東京時間フィルタとラベル作成

In [ ]:
df['hour'] = df['日時'].dt.hour
df_tokyo = df[(df['hour'] >= 8) & (df['hour'] <= 12)].copy()

df_tokyo = add_future_return_label(df_tokyo, horizon=4, threshold=0.005)
df_tokyo = add_direction_label(df_tokyo, return_col='future_return_4h', threshold=0.005)

df_tokyo[['日時', 'close', 'future_return_4h', 'is_trend', 'signal']].tail()

## 学習データ

In [ ]:
features_trend = [
    'return_1h', 'range', 'body', 'upper_wick', 'lower_wick',
    'hour', 'dayofweek', 'month', 'sma_5', 'sma_20',
    'macd', 'macd_signal', 'macd_hist', 'bb_width', 'rsi_14', 'atr_14'
]

df_model = df_tokyo.dropna(subset=features_trend + ['is_trend', 'future_return_4h']).copy()
split_index = int(len(df_model) * 0.8)

X_train = df_model[features_trend].iloc[:split_index]
X_test = df_model[features_trend].iloc[split_index:]
y_train = df_model['is_trend'].iloc[:split_index]
y_test = df_model['is_trend'].iloc[split_index:]

len(X_train), len(X_test), y_train.mean(), y_test.mean()

## LightGBM学習

In [ ]:
!pip -q install lightgbm

from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model_trend = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    class_weight='balanced',
    random_state=42,
)

model_trend.fit(X_train, y_train)
proba = model_trend.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print('Accuracy =', accuracy_score(y_test, pred))
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred))

## Direction Model

Predict whether the future 4-hour move is down, flat, or up.

In [ ]:
direction_threshold = 0.005

df_model['direction_signal'] = 0
df_model.loc[df_model['future_return_4h'] > direction_threshold, 'direction_signal'] = 1
df_model.loc[df_model['future_return_4h'] < -direction_threshold, 'direction_signal'] = -1

# Convert -1/0/1 signals to 0/1/2 classes for LightGBM multiclass training.
signal_to_class = {-1: 0, 0: 1, 1: 2}
class_to_signal = {0: -1, 1: 0, 2: 1}

y_dir = df_model['direction_signal'].map(signal_to_class)
y_dir_train = y_dir.iloc[:split_index]
y_dir_test = y_dir.iloc[split_index:]

df_model['direction_signal'].value_counts(normalize=True).sort_index()


In [ ]:
model_dir = LGBMClassifier(
    objective='multiclass',
    num_class=3,
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    class_weight='balanced',
    random_state=42,
)

model_dir.fit(X_train, y_dir_train)
dir_proba = model_dir.predict_proba(X_test)
dir_class_pred = dir_proba.argmax(axis=1)
dir_signal_pred = pd.Series(dir_class_pred, index=X_test.index).map(class_to_signal)

print('Direction Accuracy =', accuracy_score(y_dir_test, dir_class_pred))
print(confusion_matrix(y_dir_test, dir_class_pred))
print(classification_report(
    y_dir_test,
    dir_class_pred,
    target_names=['down', 'flat', 'up'],
))


## Probability-Difference Signals

Use the gap between up probability and down probability instead of taking the largest class directly. This reduces one-sided bias and keeps weak predictions out of the market.

In [ ]:
bt = df_model.iloc[split_index:].copy()
bt['trend_proba'] = proba
bt['down_proba'] = dir_proba[:, signal_to_class[-1]]
bt['flat_proba'] = dir_proba[:, signal_to_class[0]]
bt['up_proba'] = dir_proba[:, signal_to_class[1]]
bt['direction_edge'] = bt['up_proba'] - bt['down_proba']
bt['dir_confidence'] = np.maximum(bt['up_proba'], bt['down_proba'])

def make_probability_edge_signal(
    frame,
    trend_threshold=0.60,
    edge_threshold=0.20,
    min_direction_confidence=0.45,
    allow_long=True,
    allow_short=True,
):
    signal = pd.Series(0, index=frame.index, dtype=int)
    active = (
        (frame['trend_proba'] >= trend_threshold)
        & (frame['dir_confidence'] >= min_direction_confidence)
    )
    if allow_long:
        signal.loc[active & (frame['direction_edge'] >= edge_threshold)] = 1
    if allow_short:
        signal.loc[active & (frame['direction_edge'] <= -edge_threshold)] = -1
    return signal

bt[['trend_proba', 'down_proba', 'flat_proba', 'up_proba', 'direction_edge', 'dir_confidence']].describe()


## Strategy Comparison

Compare long-only, short-only, and long-short variants across several edge thresholds.

In [ ]:
strategy_rows = []
edge_thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35]

for edge_threshold in edge_thresholds:
    for name, allow_long, allow_short in [
        ('long_short', True, True),
        ('long_only', True, False),
        ('short_only', False, True),
    ]:
        signal = make_probability_edge_signal(
            bt,
            trend_threshold=0.60,
            edge_threshold=edge_threshold,
            min_direction_confidence=0.45,
            allow_long=allow_long,
            allow_short=allow_short,
        )
        returns = build_long_short_returns(
            bt.assign(trade_signal=signal),
            signal_col='trade_signal',
            return_col='future_return_4h',
            cost_per_trade=0.0002,
        )
        summary = summarize_returns(returns)
        strategy_rows.append({
            'strategy': name,
            'edge_threshold': edge_threshold,
            'long_count': int((signal == 1).sum()),
            'short_count': int((signal == -1).sum()),
            'flat_count': int((signal == 0).sum()),
            **summary,
        })

strategy_results = pd.DataFrame(strategy_rows)
display_cols = [
    'strategy', 'edge_threshold', 'long_count', 'short_count', 'trades',
    'total_return', 'entry_win_rate', 'entry_mean_return',
    'avg_win', 'avg_loss', 'profit_factor', 'max_drawdown',
]
strategy_results.sort_values('total_return', ascending=False)[display_cols]


## Selected Backtest

Start from the best row in the comparison table, then inspect recent trades. This is still a research result, not a deployable strategy.

In [ ]:
best = strategy_results.sort_values('total_return', ascending=False).iloc[0]
selected_signal = make_probability_edge_signal(
    bt,
    trend_threshold=0.60,
    edge_threshold=float(best['edge_threshold']),
    min_direction_confidence=0.45,
    allow_long=best['strategy'] in ['long_short', 'long_only'],
    allow_short=best['strategy'] in ['long_short', 'short_only'],
)
bt['trade_signal'] = selected_signal
selected_returns = build_long_short_returns(
    bt,
    signal_col='trade_signal',
    return_col='future_return_4h',
    cost_per_trade=0.0002,
)
selected_equity = equity_curve(selected_returns)
selected_yearly = yearly_returns(selected_returns, bt[df.columns[0]])

print('Selected strategy:')
print(best[display_cols])
print('\nSignal counts:')
print(bt['trade_signal'].value_counts().sort_index())
print('\nBacktest summary:')
selected_summary = summarize_returns(selected_returns)
print(selected_summary)
print('\nYearly returns:')
selected_yearly


In [ ]:
equity_preview = pd.DataFrame({
    df.columns[0]: bt[df.columns[0]].values,
    'strategy_return': selected_returns.values,
    'equity': selected_equity.values,
})
equity_preview.tail(30)


In [ ]:
datetime_col = df.columns[0]
bt[[datetime_col, 'close', 'future_return_4h', 'trend_proba', 'down_proba', 'up_proba', 'direction_edge', 'trade_signal']].tail(30)
